<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/advanced/03_llm_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Fine-Tuning & PEFT for ML Engineers

Fine-tune large language models efficiently using LoRA, QLoRA, and PEFT — from data preparation to HuggingFace Hub deployment.

**Topics:** PEFT library, LoRA/QLoRA, instruction tuning, dataset preparation, training loops, model evaluation, Hub publishing

## 1. Dataset Preparation for Instruction Tuning

In [ ]:
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer
from typing import Optional

def format_alpaca_prompt(example: dict, eos_token: str = '</s>') -> dict:
    """Format dataset example into Alpaca instruction-following prompt."""
    if example.get('input'):
        prompt = (
            f'### Instruction:\n{example["instruction"]}\n\n'
            f'### Input:\n{example["input"]}\n\n'
            f'### Response:\n{example["output"]}{eos_token}'
        )
    else:
        prompt = (
            f'### Instruction:\n{example["instruction"]}\n\n'
            f'### Response:\n{example["output"]}{eos_token}'
        )
    return {'text': prompt}

def prepare_dataset(
    dataset_name: str,
    tokenizer: AutoTokenizer,
    max_length: int = 2048,
    train_split: float = 0.9,
) -> tuple[Dataset, Dataset]:
    """Load, format, tokenize, and split a fine-tuning dataset."""
    ds = load_dataset(dataset_name, split='train')
    ds = ds.map(format_alpaca_prompt, num_proc=4)
    ds = ds.filter(lambda x: len(tokenizer.encode(x['text'])) <= max_length)
    
    split = ds.train_test_split(test_size=1 - train_split, seed=42)
    return split['train'], split['test']

# Data quality checks
import json
sample_examples = [
    {'instruction': 'Summarize the key points of the meeting.', 'input': 'We discussed Q4 budget cuts...', 'output': '1. Budget reduced by 15%\n2. Headcount freeze until Q2'},
    {'instruction': 'Translate to French.', 'input': 'Hello, how are you?', 'output': 'Bonjour, comment allez-vous?'},
]

print('Formatted training examples:')
for ex in sample_examples:
    formatted = format_alpaca_prompt(ex)
    print(formatted['text'])
    print('-' * 40)

## 2. LoRA Training with PEFT + SFTTrainer

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import BitsAndBytesConfig
from trl import SFTTrainer

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'

def load_model_for_qlora(model_name: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map='auto', trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    return model, tokenizer

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type=TaskType.CAUSAL_LM, bias='none',
)

training_args = TrainingArguments(
    output_dir='./phi3-finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,   # trade compute for memory
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    logging_steps=25,
    save_strategy='epoch',
    evaluation_strategy='epoch',
    load_best_model_at_end=True,
    report_to=['mlflow'],
    run_name='phi3-instruction-lora',
)

print('QLoRA training config:')
print(f'  Base model: {MODEL_NAME}')
print(f'  LoRA rank: {lora_config.r}')
print(f'  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  Trainable params: ~{lora_config.r * 4 * 2 * 32 / 1e6:.1f}M (for 4 target modules)')

## 3. Training Loop and Checkpoint Management

In [ ]:
import mlflow
from pathlib import Path

def train_with_callbacks(
    model, tokenizer, train_dataset, eval_dataset, lora_config, training_args,
):
    """Full training with MLflow tracking and early stopping."""
    from transformers import EarlyStoppingCallback
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()  # shows: trainable params: 4.2M || all params: 3.8B || 0.11%
    
    with mlflow.start_run(run_name=training_args.run_name):
        mlflow.log_params({
            'lora_r': lora_config.r,
            'lora_alpha': lora_config.lora_alpha,
            'learning_rate': training_args.learning_rate,
            'epochs': training_args.num_train_epochs,
        })
        
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            peft_config=lora_config,
            dataset_text_field='text',
            max_seq_length=2048,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )
        
        trainer.train()
        
        # Save adapter only (not full model)
        trainer.model.save_pretrained('./phi3-lora-adapter')
        tokenizer.save_pretrained('./phi3-lora-adapter')
        
        mlflow.log_artifact('./phi3-lora-adapter')

print('Checkpoint strategy:')
options = [
    ('Save full model', '~7GB per checkpoint', 'All weights saved'),
    ('Save LoRA adapter', '~40MB per checkpoint', 'Only ΔW = A×B saved'),
    ('Save every N steps', 'Granular recovery', 'Use save_steps=100'),
    ('Save best only', 'Disk efficient', 'Use load_best_model_at_end=True'),
]
for strategy, cost, note in options:
    print(f'  {strategy:<25}: {cost:<30} {note}')

## 4. Evaluating Fine-Tuned Models

In [ ]:
import numpy as np
from transformers import pipeline

def evaluate_model(
    model_path: str,
    test_examples: list[dict],
    base_model_path: str,
) -> dict:
    """Compare fine-tuned model vs base model on test examples."""
    from peft import PeftModel
    
    # Load both models
    base_pipe = pipeline('text-generation', model=base_model_path, device_map='auto')
    ft_model = AutoModelForCausalLM.from_pretrained(base_model_path, device_map='auto')
    ft_model = PeftModel.from_pretrained(ft_model, model_path)
    ft_pipe = pipeline('text-generation', model=ft_model, tokenizer=base_model_path)
    
    results = {'base': [], 'finetuned': []}
    for ex in test_examples:
        prompt = f'### Instruction:\n{ex["instruction"]}\n\n### Response:\n'
        base_out = base_pipe(prompt, max_new_tokens=200)[0]['generated_text']
        ft_out = ft_pipe(prompt, max_new_tokens=200)[0]['generated_text']
        results['base'].append(base_out)
        results['finetuned'].append(ft_out)
    
    return results

# Evaluation metrics for different tasks
eval_metrics = {
    'Classification': ['accuracy', 'f1', 'confusion matrix'],
    'Text generation': ['ROUGE-L', 'BLEU', 'BERTScore'],
    'Instruction following': ['LLM-as-judge (GPT-4)', 'MT-Bench', 'custom rubric'],
    'Code generation': ['pass@1', 'pass@10', 'execution accuracy'],
    'QA/RAG': ['EM (exact match)', 'F1', 'RAGAS faithfulness'],
}

print('Evaluation metrics by task type:')
for task, metrics in eval_metrics.items():
    print(f'  {task:<22}: {", ".join(metrics)}')
print()
print('Best practice: Always compare to a strong baseline (base model + few-shot).')
print('Fine-tuning should be justified by a measurable metric improvement.')

## 5. Merging and Publishing to HuggingFace Hub

In [ ]:
from peft import PeftModel
from huggingface_hub import HfApi

def merge_and_push(
    base_model_name: str,
    adapter_path: str,
    hub_model_id: str,
    hf_token: str,
):
    """Merge LoRA adapter into base model and push to HuggingFace Hub.
    
    Why merge before publishing:
    - Users don't need to know it was LoRA-trained
    - Faster inference (no PEFT overhead)
    - Compatible with any inference framework
    """
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name, torch_dtype=torch.bfloat16, device_map='cpu',
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    merged_model = model.merge_and_unload()  # fuse A×B into W, remove PEFT layer
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    
    merged_model.push_to_hub(hub_model_id, token=hf_token, private=False)
    tokenizer.push_to_hub(hub_model_id, token=hf_token)
    
    print(f'Published: https://huggingface.co/{hub_model_id}')

# Model card template
model_card_template = """
---
base_model: microsoft/Phi-3-mini-4k-instruct
license: mit
tags:
  - fine-tuned
  - lora
  - instruction-following
---

# Phi-3-mini Fine-tuned for SQL Generation

Fine-tuned on 5000 text-to-SQL examples using QLoRA (r=16).

## Performance
| Metric | Base | Fine-tuned |
|--------|------|------------|
| pass@1 | 42%  | 78%        |
| EM     | 31%  | 65%        |

## Usage
```python
from transformers import pipeline
pipe = pipeline('text-generation', model='yourname/phi3-sql')
```
"""

print('HuggingFace Hub publishing checklist:')
checklist = [
    'Write a model card (README.md) with performance metrics',
    'Include training details (dataset, LoRA config, hardware)',
    'Add license and tags for discoverability',
    'Push adapter OR merged model (never push quantized weights)',
    'Test with pipeline() before announcing',
]
for item in checklist:
    print(f'  ✓ {item}')

## Exercises

1. **Chat template:** Implement a custom chat template formatter for a model that uses `<|user|>` / `<|assistant|>` tokens. Verify that the template correctly handles multi-turn conversations and that the model only generates text after the final `<|assistant|>` token.

2. **Training curve analysis:** Plot training loss, validation loss, and learning rate over steps. Identify overfitting from the curves and modify the config to reduce it (try weight_decay, more dropout, fewer epochs).

3. **Inference optimization:** After fine-tuning, benchmark the merged model with different inference strategies: standard, Flash Attention 2, `torch.compile()`, and vLLM. Report throughput (tokens/sec) for each.